# Pitchsense — data & model exploration

Walkthrough of the dataset, the engineered features, and the trained ensemble.

In [ ]:
import sys; sys.path.insert(0, '..')
import pandas as pd, numpy as np, matplotlib.pyplot as plt
import config
from data.ingestion import load_dataset

ds = load_dataset()
m = ds.matches
print(f"{len(m):,} matches | {m.home_team.nunique()} teams | {m.date.min().date()} → {m.date.max().date()}")
m.tail()

## Goals per match over time

In [ ]:
yearly = m.assign(total=m.home_score + m.away_score, year=m.date.dt.year).groupby('year')['total'].mean()
ax = yearly.plot(figsize=(10,3.5), color='#0B6E4F')
ax.set_ylabel('avg goals / match'); ax.set_title('Scoring has settled around 2.7 goals/match'); plt.show()

## Feature matrix

In [ ]:
feat = pd.read_parquet(config.FEATURES_PARQUET)
print(feat.shape)
feat.filter(regex='elo|form10_ppg|h2h5').describe().T.head(12)

## Home advantage and Elo expectancy sanity checks

In [ ]:
recent = feat[feat.date.dt.year >= 2000]
print('home win rate (non-neutral):', (recent[recent.neutral==0].outcome==0).mean().round(3))
print('home win rate (neutral):   ', (recent[recent.neutral==1].outcome==0).mean().round(3))
bins = pd.cut(recent.elo_expectation_home, np.linspace(0,1,11))
obs = recent.groupby(bins, observed=True).apply(lambda g: (g.outcome==0).mean())
pred = recent.groupby(bins, observed=True).elo_expectation_home.mean()
plt.figure(figsize=(4.5,4.5)); plt.plot([0,1],[0,1],'--',c='grey'); plt.plot(pred, obs, 'o-', c='#0B6E4F')
plt.xlabel('Elo expectancy'); plt.ylabel('observed home result'); plt.title('Elo is well calibrated'); plt.show()

## Trained model predictions

In [ ]:
from prediction.predictor import Predictor
pred = Predictor()
p = pred.predict('Brazil', 'Germany', neutral=True)
print(p.probs, p.expected_goals)
p.top_scores[:6]